# 03 — Integrando dos conjuntos de datos

**Taller de célula única CIAD**

Adaptado de *Introduction to scRNA-seq integration* de Seurat:
<https://satijalab.org/seurat/articles/integration_introduction>

Todo hasta ahora usó una sola muestra. Los proyectos reales casi nunca son así. En
cuanto tienes dos — dos pacientes, dos condiciones, dos corridas en días distintos
— te topas con el problema del que trata este notebook.

**El problema.** Las células forman clusters según *de dónde vinieron* en lugar de
según *qué son*. Dos lotes de células B caen en dos clusters separados. Todas las
respuestas posteriores quedan mal: encuentras el número equivocado de tipos
celulares, y tu expresión diferencial encuentra el lote, no la biología.

**Los datos.** Dos conjuntos de PBMCs de 10x Genomics:

| | células | química |
|---|---|---|
| `pbmc3k` | ~2,700 | v1 (2016) |
| `pbmc_1k_v3` | ~1,200 | v3 (2018) |

Mismo tejido, mismos tipos celulares, distinta tecnología y distinto día. Ese es
un batch effect real, no simulado — y a diferencia de un conjunto con dos
condiciones, sabemos con certeza que cualquier separación por lote es técnica,
porque la biología es la misma.

**Qué hacemos**

1. construir un objeto a partir de dos conjuntos, y ver el batch effect
2. corregirlo de tres formas: **CCA de Seurat**, **Harmony** y **Canek**
3. comparar las tres, a ojo y con un número
4. preguntarnos qué nos dice y qué no nos dice esa comparación

Unos 50 minutos.

## Preparación

Este notebook usa un **archivo de preparación distinto** al de los notebooks 01 y
02.

Instala Canek además de Seurat. Canek depende de paquetes de Bioconductor, que
tardan varios minutos en instalarse, así que los notebooks anteriores
deliberadamente los dejan fuera en vez de hacer esperar a todos por algo que no
usan.

Inicia esta celda y lee la introducción de arriba mientras corre.

In [ ]:
source("https://raw.githubusercontent.com/MartinLoza/CIAD_workshop_sc/main/setup/setup_canek.R")

In [ ]:
library(harmony)

cat("Canek  ", as.character(packageVersion("Canek")), "\n")
cat("harmony", as.character(packageVersion("harmony")), "\n")

## 1. Dos conjuntos de datos

Descargamos ambos, leemos ambos, y llevamos registro de cuál es cuál.

In [ ]:
# batch 1 — the pbmc3k data from notebooks 01 and 02
download.file("https://cf.10xgenomics.com/samples/cell/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz",
              "pbmc3k.tar.gz", quiet = TRUE)
untar("pbmc3k.tar.gz")

# batch 2 — a later run, v3 chemistry
download.file("https://cf.10xgenomics.com/samples/cell-exp/3.0.0/pbmc_1k_v3/pbmc_1k_v3_filtered_feature_bc_matrix.tar.gz",
              "pbmc1k.tar.gz", quiet = TRUE)
untar("pbmc1k.tar.gz")

counts_v1 <- Read10X("filtered_gene_bc_matrices/hg19")
counts_v3 <- Read10X("filtered_feature_bc_matrix")

cat("v1:", nrow(counts_v1), "genes x", ncol(counts_v1), "cells\n")
cat("v3:", nrow(counts_v3), "genes x", ncol(counts_v3), "cells\n")

### La etiqueta de lote

La variable más importante de este notebook. Todo lo que sigue — lo que
graficamos, lo que corregimos, lo que medimos — depende de ella.

In [ ]:
v1 <- CreateSeuratObject(counts_v1, project = "v1", min.cells = 3, min.features = 200)
v3 <- CreateSeuratObject(counts_v3, project = "v3", min.cells = 3, min.features = 200)

v1$batch <- "v1_3k"
v3$batch <- "v3_1k"

cat("v1:", ncol(v1), "cells\n")
cat("v3:", ncol(v3), "cells\n")

### Control de calidad, por lote

El control de calidad se hace **dentro de cada lote**, nunca entre lotes. Las dos
químicas capturan cantidades distintas de RNA, así que un solo umbral de counts
eliminaría silenciosamente muchas más células de un lote que del otro — lo cual es
en sí mismo un batch effect que acabas de crear.

In [ ]:
v1[["percent.mt"]] <- PercentageFeatureSet(v1, pattern = "^MT-")
v3[["percent.mt"]] <- PercentageFeatureSet(v3, pattern = "^MT-")

VlnPlot(v1, features = c("nFeature_RNA", "percent.mt"), ncol = 2) +
  plot_annotation(title = "v1")


In [ ]:
VlnPlot(v3, features = c("nFeature_RNA", "percent.mt"), ncol = 2) +
  plot_annotation(title = "v3")

Mira la diferencia en `nFeature_RNA` antes de filtrar. La química v3
detecta muchos más genes por célula. Esa brecha es el batch effect, visible antes
de haber hecho cualquier análisis.

In [ ]:
before <- c(ncol(v1), ncol(v3))

v1 <- subset(v1, subset = nFeature_RNA > 200 & nFeature_RNA < 2500 & percent.mt < 5)
v3 <- subset(v3, subset = nFeature_RNA > 500 & nFeature_RNA < 5000 & percent.mt < 15)

cat("v1:", before[1], "->", ncol(v1), "cells\n")
cat("v3:", before[2], "->", ncol(v3), "cells\n")

### ✏️ Ejercicio 1

Los umbrales de arriba son distintos para los dos lotes — 200–2500 genes y 5%
mitocondrial para v1, 500–5000 y 15% para v3.

Justifica o rechaza esa decisión. Vuelve a mirar los dos violin plots. ¿Un umbral
único compartido habría sido más honesto, o menos?

No hay que programar nada. Escribe tu respuesta en la siguiente celda y prepárate
para defenderla.

*Tu respuesta:*



### Un objeto, dos layers

`merge()` combina los objetos. En Seurat v5 los counts se quedan en **layers
separadas**, una por lote — no se mezclan silenciosamente.

Esa estructura de layers es sobre la que opera `IntegrateLayers()` más adelante.

Nota primero la intersección de genes. Los dos conjuntos fueron mapeados a
versiones distintas del genoma de referencia, así que sus listas de genes
difieren. Hacer el merge sin intersectar llenaría los genes faltantes con ceros y
fabricaría una diferencia entre lotes que es puramente de contabilidad.

In [ ]:
shared <- intersect(rownames(v1), rownames(v3))

cat("genes in v1    :", nrow(v1), "\n")
cat("genes in v3    :", nrow(v3), "\n")
cat("shared         :", length(shared), "\n")

pbmc <- merge(
  v1[shared, ],
  y = v3[shared, ],
  add.cell.ids = c("v1", "v3")
)

pbmc

In [ ]:
# two layers, one per batch
Layers(pbmc[["RNA"]])

table(pbmc$batch)

## 2. Viendo el batch effect

Corremos el pipeline estándar, exactamente como en el notebook 02, y miramos el
resultado coloreado por lote.

Como las layers están separadas, la normalización y la selección de genes
variables se hacen por layer y luego se combinan — esto es Seurat v5 manejando la
estructura de lotes por ti.

In [ ]:
pbmc <- NormalizeData(pbmc, verbose = FALSE)
pbmc <- FindVariableFeatures(pbmc, verbose = FALSE)
pbmc <- ScaleData(pbmc, verbose = FALSE)
pbmc <- RunPCA(pbmc, verbose = FALSE)

pbmc <- RunUMAP(pbmc, dims = 1:30, reduction = "pca",
                reduction.name = "umap.unintegrated", verbose = FALSE)

DimPlot(pbmc, reduction = "umap.unintegrated", group.by = "batch") +
  ggtitle("no integration")

Ahí está. Las células se separan por lote, no por tipo celular.

Para asegurarnos de que eso es técnico y no biológico, revisemos un marcador.
`MS4A1` marca células B, y ambos lotes contienen células B — así que si las
células B están en dos lugares separados, la división es técnica.

In [ ]:
FeaturePlot(pbmc, reduction = "umap.unintegrated",
            features = c("MS4A1", "CD3E", "CD14", "PPBP"), ncol = 4)

Cada marcador se enciende en **dos** lugares, uno por lote. Mismo tipo
celular, partido en dos solo por la tecnología. Eso es exactamente lo que la
integración debe arreglar.

### Qué hace el clustering con esto

In [ ]:
pbmc <- FindNeighbors(pbmc, dims = 1:30, reduction = "pca", verbose = FALSE)
pbmc <- FindClusters(pbmc, resolution = 0.5, verbose = FALSE)

# how pure is each cluster, in batch terms?
round(prop.table(table(pbmc$seurat_clusters, pbmc$batch), margin = 1), 2)

Lee esa tabla así: para cada cluster, qué fracción vino de cada lote.

Un cluster de 0.98 / 0.02 es un artefacto de lote — es la versión de un tipo
celular de un solo lote. Si le pasaras estos clusters a una prueba de expresión
diferencial obtendrías una lista larga de genes significativos que en realidad son
química.

## 3. El método propio de Seurat

`IntegrateLayers()` es la puerta de entrada única de Seurat v5 para la
integración. Le pasas un método, le dices de qué reduction partir, y le pones
nombre a la reduction que va a crear. El método es intercambiable — todo lo demás
de la llamada se queda igual.

El método integrado de Seurat es **CCA**, análisis de correlación canónica. Busca
direcciones de variación que sean *compartidas* entre los lotes, bajo el argumento
de que la variación compartida es biología y la variación específica de cada lote
es técnica. Después encuentra "anchors" — pares de células de lotes distintos que
son vecinas mutuas en ese espacio compartido — y los usa para calcular la
corrección.

Este es el método que usa la viñeta de Seurat. Es el más minucioso de los tres y
también el más lento: espera un par de minutos.

In [ ]:
pbmc <- IntegrateLayers(
  object         = pbmc,
  method         = CCAIntegration,
  orig.reduction = "pca",
  new.reduction  = "integrated.cca",
  verbose        = FALSE
)

Reductions(pbmc)

In [ ]:
pbmc <- RunUMAP(pbmc, dims = 1:30, reduction = "integrated.cca",
                reduction.name = "umap.cca", verbose = FALSE)

DimPlot(pbmc, reduction = "umap.cca", group.by = "batch") +
  ggtitle("Seurat CCA")

## 4. Harmony

La misma función, distinto método — ese es el punto de `IntegrateLayers()`.

Harmony trabaja **sobre el embedding del PCA** directamente. No busca anchors
entre células: toma las coordenadas del PCA, las agrupa de forma suave, y empuja
iterativamente las células de cada lote hacia los centros de cluster compartidos,
repitiendo hasta que los lotes se traslapan.

Menos piezas móviles que CCA, y bastante más rápido.

In [ ]:
pbmc <- IntegrateLayers(
  object         = pbmc,
  method         = HarmonyIntegration,
  orig.reduction = "pca",
  new.reduction  = "harmony",
  verbose        = FALSE
)

Reductions(pbmc)

In [ ]:
pbmc <- RunUMAP(pbmc, dims = 1:30, reduction = "harmony",
                reduction.name = "umap.harmony", verbose = FALSE)

DimPlot(pbmc, reduction = "umap.harmony", group.by = "batch") +
  ggtitle("Harmony")

## 5. Canek

Un método distinto, aplicado en el mismo punto del pipeline.

Canek identifica **vecinos mutuos más cercanos** (MNN) entre lotes — pares de
células que son la coincidencia más cercana una de la otra cruzando la frontera
entre lotes, y que se asumen del mismo tipo celular. A partir de esos pares estima
la corrección, usando un híbrido de un modelo lineal y uno no lineal.

Igual que Harmony, lo corremos sobre el **embedding del PCA**, así que los dos son
directamente comparables: misma entrada, mismo tipo de salida, solo cambia la
corrección.

Dos diferencias prácticas en cómo se llama:

- Canek no es un método de `IntegrateLayers`. Se llama directamente con
  `RunCanek()`, y lee el lote de una **columna de metadata** en vez de las layers.
  Por eso unimos las layers primero.
- `correctEmbeddings = TRUE` es lo que hace que corrija el PCA en lugar de los
  valores de expresión. Lee la reduction `pca` existente y escribe una nueva
  llamada `canek`, dejando `pca` intacta — por eso todavía podemos graficar la
  versión sin corregir después.

`pcaDim = 30` corresponde a las 30 dimensiones que le dimos a Harmony. Sin eso
Canek usaría todos los componentes del PCA, y la comparación no sería pareja.

In [ ]:
pbmc[["RNA"]] <- JoinLayers(pbmc[["RNA"]])

Layers(pbmc[["RNA"]])

In [ ]:
pbmc <- RunCanek(pbmc,
                 batches           = "batch",
                 correctEmbeddings = TRUE,
                 pcaDim            = 30)

Reductions(pbmc)

Una nueva reduction, `canek`, junto a `pca` y `harmony`. No se
sobrescribió nada ni se creó un assay nuevo — la corrección vive por completo en
el embedding, exactamente igual que la de Harmony.

Así que el paso que falta es el mismo que corrimos para Harmony: un UMAP de las
coordenadas corregidas.

In [ ]:
pbmc <- RunUMAP(pbmc, dims = 1:30, reduction = "canek",
                reduction.name = "umap.canek", verbose = FALSE)

DimPlot(pbmc, reduction = "umap.canek", group.by = "batch") +
  ggtitle("Canek")

## 6. Comparando las cuatro

Lado a lado, coloreadas por lote. Lo que quieres ver: los dos colores mezclados en
todas partes, y la forma general todavía mostrando grupos distintos.

Los dos modos de fallo se ven en una figura así. Muy poca corrección deja los
lotes separados. Demasiada los funde todo en una sola mancha — lotes perfectamente
mezclados, tipos celulares destruidos.

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 5)

panel <- function(reduction, title, legend = FALSE) {
  p <- DimPlot(pbmc, reduction = reduction, group.by = "batch") + ggtitle(title)
  if (legend) p else p + theme(legend.position = "none")
}

panel("umap.unintegrated", "none") +
  panel("umap.cca",     "Seurat CCA") +
  panel("umap.harmony", "Harmony") +
  panel("umap.canek",   "Canek", legend = TRUE)

### ¿Sobrevivió la biología?

Mezclar lotes es fácil — podrías lograrlo revolviendo los datos. La prueba es si
los tipos celulares siguen siendo distinguibles después.

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 8)

FeaturePlot(pbmc, reduction = "umap.cca",
            features = c("MS4A1", "CD3E", "CD14", "PPBP"), ncol = 4) +
  plot_annotation(title = "Seurat CCA — each marker should now be in ONE place")

In [ ]:
FeaturePlot(pbmc, reduction = "umap.harmony",
            features = c("MS4A1", "CD3E", "CD14", "PPBP"), ncol = 4) +
  plot_annotation(title = "Harmony")

In [ ]:
FeaturePlot(pbmc, reduction = "umap.canek",
            features = c("MS4A1", "CD3E", "CD14", "PPBP"), ncol = 4) +
  plot_annotation(title = "Canek")

### ✏️ Ejercicio 2

Elige otro par de marcadores y revísalo en ambos UMAPs integrados. Sugerencias:
`GNLY` y `NKG7` para células NK, `FCER1A` para dendríticas, `CD8A` para T CD8.

¿El marcador queda en un solo lugar, o en dos?

In [ ]:
# FeaturePlot(pbmc, reduction = "umap.harmony", features = c(______), ncol = 2)

## 7. Ponerle un número

Los ojos no son confiables, y un UMAP es una proyección. Una métrica simple y
honesta:

> Para cada célula, mira sus 30 vecinas más cercanas en el espacio corregido. ¿Qué
> fracción viene del *otro* lote?

Si los lotes están perfectamente mezclados, esa fracción se acerca a la proporción
que el otro lote representa de los datos. Si los lotes siguen separados, se acerca
a cero.

Esta es la idea detrás de métricas publicadas como kBET y LISI, reducida a algo
que cabe en diez líneas.

In [ ]:
mixing <- function(obj, reduction, batch = "batch", k = 30, dims = 1:30) {
  emb <- Embeddings(obj, reduction)[, dims]
  b   <- as.character(obj[[batch]][, 1])

  nn <- FNN::get.knn(emb, k = k)$nn.index
  # fraction of each cell's neighbours belonging to a different batch
  mean(rowMeans(matrix(b[nn], nrow = nrow(nn)) != b))
}

# what perfect mixing would look like: the chance two random cells differ
p  <- prop.table(table(pbmc$batch))
ideal <- 1 - sum(p^2)

cat(sprintf("%-14s %s\n", "method", "cross-batch neighbours"))
cat(sprintf("%-14s %.3f\n", "none",    mixing(pbmc, "pca")))
cat(sprintf("%-14s %.3f\n", "Seurat CCA", mixing(pbmc, "integrated.cca")))
cat(sprintf("%-14s %.3f\n", "Harmony", mixing(pbmc, "harmony")))
cat(sprintf("%-14s %.3f\n", "Canek",   mixing(pbmc, "canek")))
cat(sprintf("\n%-14s %.3f  (perfectly mixed)\n", "ideal", ideal))

### Lee esto con cuidado

Más alto es más mezclado. **No** es simplemente mejor.

Un método que destruyera toda la estructura biológica sacaría un puntaje cercano
al ideal y sería inútil. El número solo te dice si los lotes se mezclaron; los
gráficos de marcadores de arriba te dicen si los tipos celulares sobrevivieron.
Necesitas ambos, y ninguno por separado es evidencia.

### ✏️ Ejercicio 3

Calcula el mismo puntaje **por tipo celular** en lugar de global.

Haz clustering sobre el embedding de Harmony, y reporta el puntaje de mezcla
dentro de cada cluster. Un cluster que sigue sin mezclarse después de integrar es
donde el método falló — y con frecuencia es un tipo celular que genuinamente solo
está presente en un lote.

Completa el espacio en blanco:

In [ ]:
pbmc <- FindNeighbors(pbmc, reduction = "harmony", dims = 1:30, verbose = FALSE)
pbmc <- FindClusters(pbmc, resolution = 0.5, verbose = FALSE)

round(prop.table(table(pbmc$seurat_clusters, pbmc$batch), margin = 1), 2)

# Which clusters are still dominated by one batch?
# threshold <- ______
# names(which(apply(prop.table(table(pbmc$seurat_clusters, pbmc$batch), 1), 1, max) > threshold))

### Clusters, antes y después

La ganancia práctica. Compara esta tabla con la de la sección 2.

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 5)

DimPlot(pbmc, reduction = "umap.harmony",
        group.by = c("batch", "seurat_clusters"))

## 8. ¿Qué método deberías usar?

Una respuesta honesta: para un batch effect directo entre corridas del mismo
tejido, la mayoría de los métodos actuales funcionan, y las diferencias que ves
aquí son más pequeñas que las diferencias causadas por tus umbrales de calidad.

Lo que sí importa:

- **Integra para visualizar y hacer clustering. Haz la expresión diferencial sobre
  los datos originales**, con el lote como covariable. Los valores corregidos han
  tenido variación removida por diseño, así que los p-values calculados sobre
  ellos no son confiables. Por eso mantuvimos intacto el assay `RNA`.
- **Verifica que la biología sobrevivió.** Siempre. Un UMAP bien mezclado no
  prueba nada por sí solo.
- **Pregúntate si deberías integrar siquiera.** Si un tipo celular está
  genuinamente presente en un lote y ausente en el otro, la integración intentará
  mezclarlo de todos modos — y puede inventar una correspondencia que no existe.

### ✏️ Ejercicio 4

Integramos dos lotes del *mismo* tejido, así que sabíamos que cualquier separación
era técnica.

Supón que en cambio los lotes fueran *control* y *tratado*, y que el tratamiento
cambiara qué tipos celulares están presentes. ¿Qué le haría la integración a esa
diferencia, y cómo la distinguirías de un batch effect?

No hay que programar nada. Esta es la pregunta que hay que pensar antes de
integrar tus propios datos.

*Tu respuesta:*



## Qué hicimos

- Construimos un objeto a partir de dos conjuntos reales y vimos el batch effect
  en los datos crudos, en el UMAP, en los marcadores y en la tabla de composición
  de clusters.
- Lo corregimos de tres formas, todas actuando sobre el embedding del PCA: CCA de
  Seurat y Harmony mediante `IntegrateLayers()`, Canek mediante
  `RunCanek(correctEmbeddings = TRUE)`.
- Los comparamos a ojo y con un puntaje de mezcla de vecinos, y dijimos
  explícitamente por qué ese puntaje no basta por sí solo.

---

### Respuestas

<details>
<summary>Haz clic para desplegar</summary>

**Ejercicio 1**

Usar umbrales distintos es lo correcto, y los violin plots son la justificación:
la química v3 detecta aproximadamente el doble de genes por célula. Un límite
superior compartido de 2,500 genes eliminaría una fracción grande de células v3
sanas y casi ninguna v1 — convirtiendo un paso de control de calidad en un batch
effect. El control de calidad pregunta "¿esto es una célula real?", y qué cuenta
como real depende de la química.

La versión honesta de "un solo umbral compartido" es un cuantil: quedarse con el
95% central *dentro de cada lote*.

**Ejercicio 2**

```r
FeaturePlot(pbmc, reduction = "umap.harmony", features = c("GNLY", "NKG7"), ncol = 2)
```

**Ejercicio 3**

```r
threshold <- 0.9
comp <- prop.table(table(pbmc$seurat_clusters, pbmc$batch), 1)
names(which(apply(comp, 1, max) > threshold))
```

Cualquier cluster que siga por encima de 0.9 después de integrar vale la pena
revisarlo directamente. A veces el método falló; a veces el tipo celular
realmente está en un solo lote, y en ese caso dejarlo sin mezclar es el
comportamiento correcto.

**Ejercicio 4**

La integración no puede distinguir entre "este lote difiere técnicamente" y "este
lote difiere biológicamente" — solo ve que los lotes difieren, y su trabajo es
eliminar eso. Si el tratamiento cambió la composición de tipos celulares, la
integración va a empujar esas células juntas y puede esconder justo el efecto que
estudias.

Formas de distinguirlos:

- Los tipos celulares compartidos por ambas condiciones deberían alinearse; una
  población genuinamente específica de una condición no debería. Si *todo* se
  alinea perfectamente, sospecha de sobrecorrección.
- Conserva los datos sin corregir y revisa si la diferencia se ve ahí.
- Haz la estadística sobre counts sin corregir, con la condición como covariable.
  La integración es para ver, no para probar.

</details>